## Datasci 530 - Project 1
### Product Scraper for St Andrews Links Online Shop

Joelle Millar
October 2026


#### Set Up

Import packages from external python script and assign global variables.

In [13]:
# Import packages from external python script
exec(open("./scripts/import_packages.py").read())

In [14]:
BASE_URL = "https://shop.standrews.com"
STARTING_URL = BASE_URL + "/homepage/"

BRAND_NAME = "J.Lindeberg"
MAX_PRODUCTS = 50           # restriction placed on collection
DELAY_SECONDS = 2           # delay between product requests
MAX_LOAD_MORE_CLICKS = 5    # pagination limit

OUTPUT_CSV = "jlindeberg_products.csv"
LOG_JSON = "collection_log.json"

# XPath selectors used to locate website elements
BRAND_XPATH = (
    '//*[@id="root"]/div/div[1]/div[1]/div/div[2]/div[3]/nav/div/ul/li[6]/span'
)
PRODUCT_LINK_XPATH = "//a[contains(@href, '/catalog/products/')]"
COOKIE_XPATH = '//*[@id="root"]/div/div[4]/div[2]/button[2]'

#### Helper Functions

In [15]:
# Check whether robots.txt permits fetching the supplied URL
def check_robots(url):
    parser = robotparser.RobotFileParser()
    parser.set_url(BASE_URL + "/robots.txt")
    parser.read()

    allowed = parser.can_fetch("*", url)
    return allowed

In [16]:
# Initialise Web Driver by starting a chrome browser
def make_driver():
    options = webdriver.ChromeOptions()
    return webdriver.Chrome(options=options)

In [17]:
# Handle the cookie consent banner
def accept_cookes(driver):
    """Click 'Accept necessary only' if the banner appears."""
    try:
        button = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable(
                (By.XPATH, COOKIE_XPATH)
            )
        )
        button.click()
        print("Accepted necessary cookies only")
    except TimeoutException:
        print("Cookie button not found, continuing")

In [18]:
# Open the brands menu from the Navigation bar and select J.Lindeberg
def open_brand(driver):
    wait = WebDriverWait(driver, 10)
    
    # Open the Brands dropdown 
    wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, BRAND_XPATH)
        )
    ).click()

    # Find the link for the selected brand

    brand_xpath = f"//a[@title='{BRAND_NAME}']"
    
    brand_link = wait.until(
        EC.presence_of_element_located(
            (By.XPATH, brand_xpath)
            )
        )

    # Click the exact brand element
    # Javascript is used because the website's menu can overlap the link
    driver.execute_script("arguments[0].click();", brand_link)

    # Wait for the brand's product links to load
    wait.until(
        EC.presence_of_element_located(
            (By.XPATH, PRODUCT_LINK_XPATH)
        )
    )

    print("Brand page: ", driver.current_url)


In [19]:
# Collect unique product URLs, with limits on pagination
def collect_product_links(driver):
    wait = WebDriverWait(driver, 10)
    wait.until(
        EC.presence_of_element_located(
            (By.XPATH, PRODUCT_LINK_XPATH)
        )
    )

    links = []

    # Collect links currently displayed in the page
    for element in driver.find_elements(By.XPATH, PRODUCT_LINK_XPATH):
        url = element.get_attribute("href")
        
        if url and url not in links:
            links.append(url)

    return links[:MAX_PRODUCTS]

In [20]:
# Return the first non-empty text found using the supplied XPath options
def first_text(driver, xpaths):
    for xpath in xpaths:
        try:
            text = driver.find_element(By.XPATH, xpath).text.strip()
            
            if text:
                return text
            
        except NoSuchElementException:
            continue
        
    return None

In [21]:
# Visit a product page and extract it's available information
def scrape_product(driver, url):
    driver.get(url)
    wait = WebDriverWait(driver, 10)

    # Wait for the product title to load
    name = wait.until(
        EC.presence_of_element_located(
            (By.XPATH, "//h1")
        )
    ).text.strip()

    # Extract fields where matching elements exist
    price = first_text(driver, [
        "//*[contains(@class,'rice')]",
        "//*[contains(text(),'£')]",
        "//*[contains(text(),'$')]",
    ])

    description = first_text(driver, [
        "//*[contains(@class,'escription')]",
        "//*[@id='description']",
    ])

    # Extract breadcrumb links to record the product's category
    breadcrumbs = driver.find_elements(
        By.XPATH, 
        "//*[contains(@class,'readcrumb')]//a "
        "| //nav[@aria-label='breadcrumb']//a"
    )

    categories = " > ".join(
        item.text.strip() 
        for item in breadcrumbs 
        if item.text.strip()
    ) or None

    return {
        "name": name,
        "vendor": BRAND_NAME,
        "price": price,
        "description": description,
        "categories": categories,
        "url": url,
        "scraped_at": datetime.now().isoformat(timespec="seconds"),
    }

### Compliance Check

In [22]:
allowed = check_robots(STARTING_URL)

if not check_robots('BASE_URL + "/catalog/products/"'):
    raise SystemExit("robots.txt disallows this product path")

print("robots.txt permits the product path")

robots.txt permits the product path


### Start Browser, open homepage, accept cookies

In [23]:
# Record the start time and initialise the output lists
started = datetime.now().isoformat(timespec="seconds")
records, errors = [], []

driver = make_driver()

try:
    # Open the website and dismiss the cookie banner        
    driver.get(STARTING_URL)
    print(driver.title, "|", driver.current_url)
    
    accept_cookes(driver)

    # Navigate to the selected brand
    open_brand(driver)

    # Collect product URLs
    product_links = collect_product_links(driver)
    print("Products to visit: ", len(product_links))

    # Visit each product page and save its extracted information
    for i, url in enumerate(product_links, start=1):
        try:
            record = scrape_product(driver, url)
            records.append(record)

            print(
                f"[{i}/{len(product_links)}] "
                f"{record['name']} | {record['price']}"
            )

        except Exception as error:
            errors.append({
                "url": url, 
                "error": type(error).__name__,
                "message": str(error),
            })

            print(f"Skipped product: {type(error).__name__}")
        
        # Pause between product requests
        time.sleep(DELAY_SECONDS)

finally:
    # Close Chrome even if the scraper encounters an error
    driver.quit()

St Andrews Links : The Home of Golf | https://shop.standrews.com/homepage/
Accepted necessary cookies only
Brand page:  https://shop.standrews.com/catalog/category/jlindeberg
Products to visit:  11
[1/11] J.Lindeberg Heyden Quilt Hybrid Jacket | Free Shipping On Orders Over £175 (EUR) or £250 (USA/CAN)
[2/11] J.Lindeberg Heath Polo | £120.00
[3/11] J.Lindeberg Tour Tech Polo | Free Shipping On Orders Over £175 (EUR) or £250 (USA/CAN)
[4/11] J.Lindeberg Tour Tech Polo | Free Shipping On Orders Over £175 (EUR) or £250 (USA/CAN)
[5/11] J.Lindeberg Tour Tech Print Polo | Free Shipping On Orders Over £175 (EUR) or £250 (USA/CAN)
[6/11] J.Lindeberg Lukas 1/4 Zip Midlayer | Free Shipping On Orders Over £175 (EUR) or £250 (USA/CAN)
[7/11] J.Lindeberg Sole Hoodie | Free Shipping On Orders Over £175 (EUR) or £250 (USA/CAN)
[8/11] J.Lindeberg Club Hoodie | £180.00
[9/11] J.Lindeberg Heath Polo | Free Shipping On Orders Over £175 (EUR) or £250 (USA/CAN)
[10/11] J.Lindeberg Heyden Quilt Hybrid Jack

### Close browser, save data and log products

In [24]:
# Convert the collected records into a DataFrame
df = pd.DataFrame(records)

# Save the product data as a CSV file
df.to_csv(OUTPUT_CSV, index=False)

# Record how the collection was conducted
log = {
    "website": BASE_URL,
    "brand": BRAND_NAME,
    "collection_started": started,
    "collection_finished": datetime.now().isoformat(timespec="seconds"),
    "observations_collected": len(df),
    "pages_skipped": len(errors),
    "restrictions": {
        "max_products": MAX_PRODUCTS,
        "delay_seconds_between_requests": DELAY_SECONDS,
        "max_load_more_clicks": MAX_LOAD_MORE_CLICKS,
    },
    "robots_check": "Allowed" if check_robots(
        BASE_URL + "/catalog/products/"
    ) else "Disallowed",
    "errors": errors,
}

with open(LOG_JSON, "w") as file:
    json.dump(log, file, indent=2)

print(f"Saved {len(df)} products to {OUTPUT_CSV}")
print(f"Saved collection log to {LOG_JSON}")

display(df.head())
print(json.dumps(log, indent=2))

Saved 11 products to jlindeberg_products.csv
Saved collection log to collection_log.json


,name,vendor,price,description,categories,url,scraped_at
0,J.Lindeberg Heyden Quilt Hybrid Jacket,J.Lindeberg,Free Shipping On Orders Over £175 (EUR) or £25...,Every purchase helps preserve the future of th...,Home > J.Lindeberg Heyden Quilt Hybrid Jacket,https://shop.standrews.com/catalog/products/j_...,2026-10-19T10:25:59
1,J.Lindeberg Heath Polo,J.Lindeberg,£120.00,Every purchase helps preserve the future of th...,Home > J.Lindeberg Heath Polo,https://shop.standrews.com/catalog/products/j_...,2026-10-19T10:26:04
2,J.Lindeberg Tour Tech Polo,J.Lindeberg,Free Shipping On Orders Over £175 (EUR) or £25...,Every purchase helps preserve the future of th...,Home > J.Lindeberg Tour Tech Polo,https://shop.standrews.com/catalog/products/j_...,2026-10-19T10:26:07
3,J.Lindeberg Tour Tech Polo,J.Lindeberg,Free Shipping On Orders Over £175 (EUR) or £25...,Every purchase helps preserve the future of th...,Home > J.Lindeberg Tour Tech Polo,https://shop.standrews.com/catalog/products/j_...,2026-10-19T10:26:12
4,J.Lindeberg Tour Tech Print Polo,J.Lindeberg,Free Shipping On Orders Over £175 (EUR) or £25...,Every purchase helps preserve the future of th...,Home > J.Lindeberg Tour Tech Print Polo,https://shop.standrews.com/catalog/products/j_...,2026-10-19T10:26:16


{
  "website": "https://shop.standrews.com",
  "brand": "J.Lindeberg",
  "collection_started": "2026-10-19T10:25:50",
  "collection_finished": "2026-10-19T10:26:41",
  "observations_collected": 11,
  "pages_skipped": 0,
  "restrictions": {
    "max_products": 50,
    "delay_seconds_between_requests": 2,
    "max_load_more_clicks": 5
  },
  "robots_check": "Allowed",
  "errors": []
}
